In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

dbutils.widgets.text("recovery_mode", "")
dbutils.widgets.text("file_id", "")
dbutils.widgets.text("recovery_run_id", "")

recovery_mode = dbutils.widgets.get("recovery_mode").strip().upper()
file_id = dbutils.widgets.get("file_id").strip()
recovery_run_id = dbutils.widgets.get("recovery_run_id").strip()

if recovery_mode != "EXECUTE":
    raise ValueError("Only EXECUTE mode is supported")

if not file_id or not recovery_run_id:
    raise ValueError("Missing recovery parameters")

print("Recovery mode:", recovery_mode)
print("File ID:", file_id)
print("Recovery Run ID:", recovery_run_id)


In [0]:

# Verify that this recovery execution owns the file

control_rows = spark.sql("""
SELECT file_id, batch_id, processing_stage,
       pipeline_status, retry_count, adf_run_id
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = :file_id
  AND adf_run_id = :run_id
  AND processing_stage = 'SILVER'
  AND pipeline_status = 'IN_PROGRESS'
  AND is_enabled = TRUE
""", args={
    "file_id": file_id,
    "run_id": recovery_run_id
}).collect()

if len(control_rows) != 1:
    raise RuntimeError("Recovery ownership validation failed")

batch_id = control_rows[0]["batch_id"]

# Reuse existing Silver audit attempt 2
audit_rows = spark.sql("""
SELECT audit_id, attempt_number
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE file_id = :file_id
  AND adf_run_id = :run_id
  AND processing_stage = 'SILVER'
  AND stage_status = 'IN_PROGRESS'
  AND stage_end_ts IS NULL
""", args={
    "file_id": file_id,
    "run_id": recovery_run_id
}).collect()

if len(audit_rows) != 1:
    raise RuntimeError("Expected one active recovery audit")

silver_audit_id = audit_rows[0]["audit_id"]
if audit_rows[0]["attempt_number"] != 2:
    raise RuntimeError(
        "Unexpected Silver recovery attempt. "
        "Expected attempt 2 for Batch 7."
    )

print("Recovery validation PASSED")
print("Batch ID:", batch_id)
print("Audit ID:", silver_audit_id)
print("Attempt:", audit_rows[0]["attempt_number"])


In [0]:

# Validate Bronze source and Silver state

bronze_batch = (
    spark.table(f"{catalog}.bronze.patients_raw")
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("batch_id") == batch_id)
    )
)

silver = spark.table(f"{catalog}.silver.DimPatients")

bronze_count = bronze_batch.count()

invalid_count = bronze_batch.filter(
    F.col("patient_id").isNull() |
    (F.trim(F.col("patient_id")) == "")
).count()

duplicate_count = (
    bronze_batch.groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_batch_count = (
    silver.filter(F.col("BatchId") == batch_id)
    .count()
)

print("Bronze records:", bronze_count)
print("Invalid patient IDs:", invalid_count)
print("Duplicate patient IDs:", duplicate_count)
print("Existing Silver batch records:", silver_batch_count)

if bronze_count == 0:
    raise RuntimeError("No Bronze records found")

if invalid_count > 0 or duplicate_count > 0:
    raise RuntimeError("Bronze data quality validation failed")

if silver_batch_count > 0:
    raise RuntimeError(
        "Possible previous Silver writes detected. "
        "Reconciliation required before recovery."
    )

# Verify existing patient history is structurally valid
patient_ids = bronze_batch.select("patient_id").distinct()

history_check = (
    silver.join(
        patient_ids,
        silver.PatientId == patient_ids.patient_id,
        "inner"
    )
    .groupBy("PatientId")
    .agg(
        F.sum(
            F.when(F.col("IsActive") == True, 1).otherwise(0)
        ).alias("active_count"),
        F.sum(
            F.when(
                (F.col("IsActive") == False) &
                F.col("RowEndDate").isNull(),
                1
            ).otherwise(0)
        ).alias("invalid_closed_count")
    )
)

unsafe_history = history_check.filter(
    (F.col("active_count") != 1) |
    (F.col("invalid_closed_count") > 0)
).count()

if unsafe_history > 0:
    raise RuntimeError(
        "Silver history requires reconciliation"
    )

print("Recovery source validation PASSED")


In [0]:

# Execute Silver transformation
# Do not automatically retry if execution becomes uncertain.

try:
    result = spark.sql("""
        CALL dbw_healthcare_de_dev.silver.usp_LoadDimPatients(
            :file_id
        )
    """, args={
        "file_id": file_id
    }).collect()

    print("Silver stored procedure completed")

    for row in result:
        print(row)

except Exception as error:
    print("SILVER RECOVERY FAILED")
    print("File ID:", file_id)
    print("Audit ID:", silver_audit_id)
    print("Error:", str(error))

    # The stored procedure may have partially committed.
    # Preserve audit IN_PROGRESS for reconciliation.
    raise RuntimeError(
        "Silver recovery failed. "
        "Do not automatically retry this execution. "
        f"Audit ID: {silver_audit_id}"
    ) from error


In [0]:

# Post-load validation and audit completion

validation = spark.sql("""
SELECT
    COUNT(*) AS source_count,
    SUM(
        CASE
            WHEN D.PatientId IS NULL THEN 1
            WHEN NOT (B.first_name <=> D.FirstName) THEN 1
            WHEN NOT (B.last_name <=> D.LastName) THEN 1
            WHEN NOT (B.gender <=> D.Gender) THEN 1
            WHEN NOT (B.date_of_birth <=> D.DateOfBirth) THEN 1
            WHEN NOT (B.state <=> D.State) THEN 1
            ELSE 0
        END
    ) AS mismatched_records
FROM dbw_healthcare_de_dev.bronze.patients_raw B
LEFT JOIN dbw_healthcare_de_dev.silver.DimPatients D
    ON B.patient_id = D.PatientId
   AND D.IsActive = TRUE
WHERE B.file_id = :file_id
  AND B.batch_id = :batch_id
""", args={
    "file_id": file_id,
    "batch_id": batch_id
}).first()

if (
    validation["source_count"] != bronze_count
    or validation["mismatched_records"] != 0
):
    raise RuntimeError(
        "Silver source-to-target validation failed. "
        "Reconciliation required."
    )

scd_errors = spark.sql("""
SELECT B.patient_id, COUNT(D.DimPatientID) AS active_count
FROM (
    SELECT DISTINCT patient_id
    FROM dbw_healthcare_de_dev.bronze.patients_raw
    WHERE file_id = :file_id
      AND batch_id = :batch_id
) B
LEFT JOIN dbw_healthcare_de_dev.silver.DimPatients D
    ON B.patient_id = D.PatientId
   AND D.IsActive = TRUE
GROUP BY B.patient_id
HAVING COUNT(D.DimPatientID) <> 1
""", args={
    "file_id": file_id,
    "batch_id": batch_id
}).collect()

if scd_errors:
    raise RuntimeError(
        f"SCD validation failed: {scd_errors}"
    )

print("Silver data validation PASSED")

# Close the existing recovery audit, not a new audit
spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
    'SUCCESS',
    :audit_id,
    :file_id,
    :run_id,
    'SILVER',
    :source_count,
    :processed_count,
    0,
    NULL
)
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "run_id": recovery_run_id,
    "source_count": bronze_count,
    "processed_count": bronze_count
}).collect()

audit_result = spark.sql("""
SELECT stage_status, stage_end_ts
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE audit_id = :audit_id
  AND file_id = :file_id
  AND adf_run_id = :run_id
  AND processing_stage = 'SILVER'
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "run_id": recovery_run_id
}).collect()

if (
    len(audit_result) != 1
    or audit_result[0]["stage_status"] != "SUCCESS"
    or audit_result[0]["stage_end_ts"] is None
):
    raise RuntimeError(
        "Silver audit SUCCESS verification failed"
    )

print("Silver recovery audit completed successfully")


In [0]:

# Advance file control to GOLD

spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_update_file_stage(
    :file_id,
    :run_id,
    'SILVER'
)
""", args={
    "file_id": file_id,
    "run_id": recovery_run_id
}).collect()

stage_rows = spark.sql("""
SELECT processing_stage, pipeline_status
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = :file_id
  AND adf_run_id = :run_id
""", args={
    "file_id": file_id,
    "run_id": recovery_run_id
}).collect()

if (
    len(stage_rows) != 1
    or stage_rows[0]["processing_stage"] != "GOLD"
    or stage_rows[0]["pipeline_status"] != "IN_PROGRESS"
):
    raise RuntimeError(
        "Silver-to-Gold transition verification failed"
    )

print("SILVER RECOVERY SUCCESS")
print("File ID:", file_id)
print("Audit ID:", silver_audit_id)
print("Records validated:", bronze_count)
print("Current stage: GOLD")
